#### 1. Librerías.

In [97]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [98]:
#a. Modo de ejecución (se define en ./constantes/modo.txt, un solo lugar para los 4 notebooks).
# "validacion" = entreno solo con train, puedo medir nDCG.
# "entrega"    = entreno con train+test, uso todo el historial para predecir.
with open("./constantes/modo.txt") as f:
    MODO = f.read().strip()

assert MODO in ("validacion", "entrega"), f"MODO inválido: {MODO!r}"
print(f"MODO: {MODO}")

MODO: entrega


In [99]:
#b. Otras constantes.
%run "./constantes/constantes.ipynb"

#### 3. Funciones.

In [100]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [101]:
#a. Train.
df_train = pd.read_csv(path_train_fe, dtype={"id_lector": str, "id_libro": str})

In [102]:
#b. Test (solo hace falta para evaluar; en entrega ya está dentro de la base).
if MODO == "validacion":
    df_test = pd.read_csv(path_test_crudo, dtype={"id_lector": str, "id_libro": str})

In [103]:
#c. Dataset a predecir.
df_a_predecir = pd.read_csv(path_a_predecir, dtype={"id_lector": str})

In [104]:
#d. Libros y Lectores.
df_libros = pd.read_csv(path_libros_fe, dtype={"id_libro": str})
df_lectores = pd.read_csv(path_lectores_fe, dtype={"id_lector": str})

#### 5. Preparación previa.

In [105]:
#a. Forma final.
print(f"Modo: {MODO}")
print(f"Train: {df_train.shape}")
if MODO == "validacion":
    print(f"Test:  {df_test.shape}")

Modo: entrega
Train: (461407, 133)


In [106]:
#b. Me aseguro que no hayan quedado nulos en los ratings.
cols_rating = [c for c in df_train.columns if c.startswith("rating_prom")]
print("Train.")
print(df_train[cols_rating].isna().sum())

Train.
rating_prom_id_lector                          0
rating_prom_id_lector_autor                    0
rating_prom_id_lector_genero_libro_agrupado    0
rating_prom_id_libro                           0
rating_prom_autor                              0
rating_prom_genero                             0
dtype: int64


In [107]:
#c. Defino las features del modelo.
features_base = [
    "anio_edicion", 
    "nacimiento",
    #"edad_al_interactuar",            # En train varía con fecha.dt.year, al predecir es
    #"dias_transcurridos_interaccion", # anio_actual - nacimiento (constante por lector).
    #"anios_transcurridos_edicion",    # Idem: al predecir queda determinada por anio_edicion.
    #"antiguedad_libro_hoy",
    'frecuencia_lector', 
    'frecuencia_libro', 
    'n_lectores_distintos_autor',
    'n_interacciones_lector_autor',   # reemplazada por prop_lector_autor (dan el mismo nCDG).
    'n_interacciones_lector_genero',  # reemplazada por prop_lector_genero (dan el mismo nCDG).
    'n_autores_distintos_lector',     # reemplazada por prop_autores_distintos (dan el mismo nCDG).
    'n_generos_distintos_lector',     # reemplazada por prop_generos_distintos (dan el mismo nCDG).
    #'rating_prom_id_lector', 
    'rating_prom_id_lector_autor',
    'rating_prom_id_lector_genero_libro_agrupado', 
    # Las tres de calidad de ítem degradaban el ranking con RF sobre RMSE
    # (0.0697 -> 0.0409). Con LGBM Ranker pasa algo parecido:
    # La mejor feature para predecir rating es la peor para rankear, 
    # y cambiar el objetivo mitiga pero no elimina el conflicto.
    #'rating_prom_id_libro',
    #'rating_prom_autor',
    #'rating_prom_genero',
    #'prop_lector_autor',
    #'prop_lector_genero',
    #'prop_autores_distintos',
    #'prop_generos_distintos',
    #'log_pop_media_lector',   # No mejora el nCDG ni lo empeora.
    #'dif_log_pop',            # No mejora el nCDG ni lo empeora.
    'score_ease',   # score colaborativo de EASE^R
    'score_ease_reciente', # score colaborativo de EASE^R pero calculado sobre los libros recientes del lector.
    'dias_desde_ultima_lectura',   # constante por lector: contexto
    #'anio_ultima_lectura',         # idem
    'antiguedad_relativa',         # esta sí varía entre candidatos
]
features_dummies = [c for c in df_train.columns if c.startswith((
    "genero_persona_", 
    #"genero_libro_agrupado_", 
    #"editorial_agrupada_", 
    #"pais_agrupado_"
))]
features = features_base + features_dummies

print(f"{len(features)} features:", features)

18 features: ['anio_edicion', 'nacimiento', 'frecuencia_lector', 'frecuencia_libro', 'n_lectores_distintos_autor', 'n_interacciones_lector_autor', 'n_interacciones_lector_genero', 'n_autores_distintos_lector', 'n_generos_distintos_lector', 'rating_prom_id_lector_autor', 'rating_prom_id_lector_genero_libro_agrupado', 'score_ease', 'score_ease_reciente', 'dias_desde_ultima_lectura', 'antiguedad_relativa', 'genero_persona_-', 'genero_persona_hombre', 'genero_persona_mujer']


#### 6. Tablas de referencia y universo de candidatos.

In [108]:
#a. Tablas de referencia para el feature engineering de los candidatos.
# Van ANTES del entrenamiento porque el ranker las necesita para armarle las features
# a los negativos. feature_engineering_test las toma como globales, así que los nombres
# del desempaquetado tienen que quedar exactamente así.
(media_global, caract_lector, caract_libros,
 afinidad_lector_autor_test, afinidad_lector_genero_test) = armar_tablas_referencia(
    df_train, df_libros, df_lectores
)

Nulos caract_lector: ninguno
Nulos caract_libros: {'autor': 78227, 'genero_libro_agrupado': 78220}
Duplicados (lector / libro / lector-autor / lector-genero): 0 0 0 0
Lectores: 11285 | Libros: 128743 | Libros sin interacciones en la base: 80676


In [109]:
#b. Universo de libros candidatos.
#i. Todos los libros con al menos una interacción.
conn = sqlite3.connect(path_db)
todos_los_libros = pd.read_sql("SELECT id_libro FROM interacciones", conn)["id_libro"].unique()
conn.close()

#ii. Filtro los libros sin la metadata que necesitan las features. Dos casos:
# - ~70 libros que están en interacciones pero no en df_libros (no tienen fila).
# - ~4 libros que sí están pero con autor en NaN: el merge de afinidad se hace sobre
#   ["id_lector", "autor"], y NaN nunca matchea contra NaN, así que las features de
#   afinidad quedan sin calcular (ni siquiera llegan al fillna).
n_antes = len(todos_los_libros)
libros_con_metadata = set(
    caract_libros.dropna(subset=["autor", "genero_libro_agrupado", "anio_edicion"])["id_libro"]
)
todos_los_libros = np.array([b for b in todos_los_libros if b in libros_con_metadata])

print(f"Universo de candidatos: {len(todos_los_libros):,} "
      f"(descarto {n_antes - len(todos_los_libros)} sin metadata)")

Universo de candidatos: 48,062 (descarto 75 sin metadata)


In [110]:
#c. Historial por lector (lo usa retrieval para excluir lo ya leído).
leidos_por_lector = (
    df_train[["id_lector", "id_libro"]]
    .groupby("id_lector")["id_libro"]
    .apply(set)
    .to_dict()
)
print(f"Lectores con historial: {len(leidos_por_lector):,}")

Lectores con historial: 10,673


In [111]:
#d. Historial ordenado por fecha, para el score_ease reciente.
# df_train ya viene ordenado por (id_lector, fecha) desde el notebook 1.
N_RECIENTES = 30
leidos_recientes_por_lector = (
    df_train.groupby("id_lector")["id_libro"]
    .apply(lambda s: list(s)[-N_RECIENTES:])
    .to_dict()
)

In [112]:
#e. Entreno EASE^R.
# min_interacciones=2 -> 23.693 items, matriz de 2.2 GB en float32. El pico de la
# inversión ronda los 7 GB. Si tira MemoryError, subir a 3 (16.527 items, 1.1 GB).
MIN_INTERACCIONES_EASE = 2
LAMBDA_EASE = 250.0

B_ease, libros_idx_ease, lectores_idx_ease, X_ease = entrenar_ease(
    df_train,
    min_interacciones=MIN_INTERACCIONES_EASE,
    lambda_reg=LAMBDA_EASE
)

Items: 26,508 | Lectores: 10,464 | Interacciones: 439,778
Matriz item x item: 2.81 GB (float32), pico estimado en la inversion: ~8.4 GB
Calculando X^T X ...
Invirtiendo ...
Listo. B: (26508, 26508), 2.81 GB


In [113]:
#f. Exporto EASE para reusarlo en el notebook 4.
if MODO == "entrega":
    np.save(path_ease_B, B_ease)
    joblib.dump(libros_idx_ease, path_ease_idx)
    print(f"EASE exportado: {path_ease_B} ({B_ease.nbytes / 1e9:.2f} GB)")
else:
    print("Salteo. Estamos en modo validación.")

EASE exportado: ./modelos/ease_B_entrega.npy (2.81 GB)


#### 7. Dataset de entrenamiento del ranker.

El RF entrenaba solo con positivos (pares lector-libro que existieron) porque predecía el
rating. Un ranker aprende a **ordenar** dentro de cada lector, así que necesita ver también
lo que el lector NO eligió: si todos los ejemplos de un lector son cosas que le gustaron,
no hay nada que ordenar.

In [114]:
#a. Parámetros del muestreo de negativos.
N_POR_POSITIVO = 5      # negativos por cada libro leído del lector
MAX_NEG_POR_LECTOR = 4000   # tope: grupo máximo = positivos + 4000, bajo el límite de 10.000
SESGO_POPULARIDAD = 0.0   # 0.0 (uniforme). Con sesgo, los negativos se concentran
                          # en libros populares: el lector pudo haberlos leído y no lo hizo,
                          # así que el modelo tiene que aprender algo más fino que
                          # "popular = bueno". Con negativos uniformes casi todos caen en
                          # la cola larga y son triviales de descartar.
N_RETRIEVAL = 2000   # tiene que coincidir con el del loop de evaluación

In [115]:
#b. Muestreo los negativos.
# Excluyo los pocos lectores que están en interacciones pero no en df_lectores: sin metadata
# no se les pueden armar features, y nunca van a estar en df_a_predecir.
lectores_con_metadata = set(caract_lector["id_lector"])
df_base_neg = df_train[df_train["id_lector"].isin(lectores_con_metadata)]

if len(df_base_neg) < len(df_train):
    print(f"Excluyo {df_train['id_lector'].nunique() - df_base_neg['id_lector'].nunique()} "
          f"lectores sin metadata ({len(df_train) - len(df_base_neg)} filas).\n")

# Negativos "duros": muestreados del top-N de EASE en vez de uniformes sobre el catálogo.
# El ranker tiene que aprender a ordenar candidatos plausibles, que es lo que va a ver
# en predicción — no "plausible vs cualquier libro al azar".
df_negativos = muestrear_negativos_ease(
    df_base_neg, todos_los_libros,
    B_ease, libros_idx_ease, leidos_por_lector,
    n_retrieval=N_RETRIEVAL,
    n_por_positivo=N_POR_POSITIVO,
    max_negativos_por_lector=MAX_NEG_POR_LECTOR
)

Excluyo 6 lectores sin metadata (122 filas).

  2,000/10,667
  4,000/10,667
  6,000/10,667
  8,000/10,667
  10,000/10,667
Negativos generados: 2,109,730
Ratio real:          4.57
Grupo mas grande:    4,260  <-- < 10.000
Solapamiento con positivos: 0  <-- 0


In [116]:
#c. Features de los negativos.
# feature_engineering_test procesa un lector por vez (sirve para el loop de predicción);
# acá son ~2M de pares, así que uso la versión en batch: mismos merges, DataFrame entero.
df_neg_fe = feature_engineering_batch(
    df_negativos,
    caract_lector, caract_libros,
    afinidad_lector_autor_test, afinidad_lector_genero_test,
    media_global
)
df_neg_fe["relevancia"] = 0

print(f"Negativos con features: {df_neg_fe.shape}")

Negativos con features: (2109730, 69)


In [117]:
#d. Relevancia de los positivos, derivada del rating.
# Filtro también por libro: hay ~43 libros que la gente leyó y puntuó pero que no están
# en el catálogo (df_libros). Sin metadata no tienen features, así que no sirven para
# entrenar y además nunca van a poder ser recomendados (no están en todos_los_libros).
CORTE_ALTO, CORTE_MEDIO = 9, 7

df_pos_fe = df_train[
    df_train["id_lector"].isin(lectores_con_metadata)
    & df_train["id_libro"].isin(libros_con_metadata)
].copy()

n_descartados = len(df_train) - len(df_pos_fe)
if n_descartados:
    print(f"Descarto {n_descartados} positivos sin metadata de lector o libro.\n")

df_pos_fe["relevancia"] = np.where(
    df_pos_fe["rating"] >= CORTE_ALTO, 3,
    np.where(df_pos_fe["rating"] >= CORTE_MEDIO, 2, 1)
)

print(df_pos_fe.groupby("relevancia")["rating"].agg(["min", "max", "count"]))

Descarto 339 positivos sin metadata de lector o libro.

            min  max   count
relevancia                  
1             1    6  146483
2             7    8  207155
3             9   10  107430


In [118]:
#e. score_ease para los positivos.
# Los negativos ya lo traen de feature_engineering_batch; los positivos vienen de df_train
# (notebook 2), que no tiene esta columna. Sin esto quedan en NaN y salta el assert.
df_pos_fe["score_ease"] = 0.0
df_pos_fe["score_ease_reciente"] = 0.0
for lector, g in df_pos_fe.groupby("id_lector", sort=False):
    libros = g["id_libro"].tolist()
    df_pos_fe.loc[g.index, "score_ease"] = score_ease(
        lector, libros, B_ease, libros_idx_ease, leidos_por_lector
    )
    df_pos_fe.loc[g.index, "score_ease_reciente"] = score_ease(
        lector, libros, B_ease, libros_idx_ease, leidos_recientes_por_lector
    )

In [119]:
#f. Junto positivos y negativos, y ordeno por lector.
# El orden por id_lector NO es cosmético: LightGBM arma los grupos a partir de bloques
# contiguos de filas. Si no está ordenado, mezcla lectores dentro de un grupo y falla
# en silencio (entrena, pero aprende a ordenar entre lectores distintos).
cols_necesarias = ["id_lector", "id_libro", "relevancia"] + features

df_ranker = pd.concat(
    [df_pos_fe[cols_necesarias], df_neg_fe[cols_necesarias]],
    ignore_index=True
)
df_ranker = df_ranker.sort_values("id_lector", kind="mergesort").reset_index(drop=True)

print(f"Filas: {len(df_ranker):,}")
print(df_ranker["relevancia"].value_counts().sort_index())

Filas: 2,570,798
relevancia
0    2109730
1     146483
2     207155
3     107430
Name: count, dtype: int64


In [120]:
#g. Comprobaciones antes de entrenar.
#i. No puede quedar ningún nulo: LightGBM los acepta, pero un NaN acá sería un merge fallido,
# no un dato faltante legítimo.
nulos = df_ranker[features].isna().sum()
assert nulos.sum() == 0, f"NaN en features: {nulos[nulos > 0].to_dict()}"
print("Nulos en features: ninguno")

#ii. El orden por lector tiene que ser estricto (si no, los grupos quedan mal).
assert df_ranker["id_lector"].is_monotonic_increasing, "df_ranker no está ordenado por lector"
print("Orden por lector: ok")

#iii. Ningún lector puede tener solo negativos (no habría nada que ordenar).
por_lector = df_ranker.groupby("id_lector")["relevancia"].max()
print(f"Lectores sin ningún positivo: {(por_lector == 0).sum()}")

Nulos en features: ninguno
Orden por lector: ok
Lectores sin ningún positivo: 0


#### 8. Entrenamiento.

In [121]:
#a. Armo X, y, y el vector de grupos.
# group es la cantidad de filas de cada lector, en el orden en que aparecen. Es lo que le
# dice a LightGBM dónde empieza y termina cada ranking.
X = df_ranker[features]
y = df_ranker["relevancia"]
group = df_ranker.groupby("id_lector", sort=False).size().values

assert group.sum() == len(X), "El vector de grupos no cubre todas las filas"
print(f"X: {X.shape} | grupos: {len(group):,} | filas por grupo: "
      f"min {group.min()}, mediana {int(np.median(group))}, max {group.max()}")

X: (2570798, 18) | grupos: 10,667 | filas por grupo: min 6, mediana 54, max 4260


In [122]:
#b. Entreno el ranker.
# objective="lambdarank" + eval_at=[20] optimiza directamente nDCG@20: el gradiente pondera
# cada par por cuánto cambiaría la métrica al intercambiarlos, así que un error en las
# primeras 20 posiciones pesa muchísimo más que uno en la 8.000. Es exactamente lo que el
# RF sobre RMSE no hacía.
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    eval_at=[20],
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

ranker.fit(X, y, group=group)
print("Entrenado.")

Entrenado.


In [123]:
#c. Importancia de las features (para ver en qué se apoya).
imp = pd.Series(ranker.feature_importances_, index=features).sort_values(ascending=False)
print(imp.head(15))

rating_prom_id_lector_autor                    3941
frecuencia_libro                               3040
n_lectores_distintos_autor                     2953
rating_prom_id_lector_genero_libro_agrupado    2819
frecuencia_lector                              2561
score_ease                                     2141
n_autores_distintos_lector                     1858
anio_edicion                                   1619
nacimiento                                     1549
antiguedad_relativa                            1483
n_interacciones_lector_autor                   1475
score_ease_reciente                            1449
n_interacciones_lector_genero                  1350
dias_desde_ultima_lectura                      1201
n_generos_distintos_lector                     1130
dtype: int32


In [124]:
#d. Exportamos el modelo entrenado.
joblib.dump(ranker, path_modelo_lgbmranker_con_ease)
print(f"Modelo exportado con exito en: {path_modelo_lgbmranker_con_ease}")

Modelo exportado con exito en: ./modelos/modelo_lgbmranker_con_ease_entrega.pkl


In [125]:
#e. En modo entrega el notebook termina aca: lo que sigue es evaluacion,
# y no tiene sentido evaluar contra un test que ya esta dentro del entrenamiento.
if MODO != "validacion":
    print("\nModo entrega: fin del notebook 3. Segui con el notebook 4.")
    raise KeyboardInterrupt("Corte intencional: modo entrega.")


Modo entrega: fin del notebook 3. Segui con el notebook 4.


KeyboardInterrupt: Corte intencional: modo entrega.

#### 9. Evaluación del modelo sobre test - nDCG@20.

In [ ]:
#a. Ground truth por lector (por diseño, 20 libros cada uno).
gt_por_lector = {lid: pd.Series(g["rating"].values, index=g["id_libro"].values)
                 for lid, g in df_test.groupby("id_lector")}
print(f"Lectores en test: {len(gt_por_lector):,}")

Lectores en test: 3,893


In [ ]:
#b. Genero una muestra de test estratificada por frecuencia, para que la composición
# se parezca a la de los lectores que evalúa Kaggle. Sin esto optimizo para lectores
# livianos (mediana 45) y me miden sobre una población más pesada (mediana 72).
freq_train = df_train.groupby("id_lector").size()
bins = [-1, 0, 20, 50, 100, 250, np.inf]

#i. Distribución objetivo: la de los lectores a predecir.
freq_objetivo = df_a_predecir["id_lector"].map(freq_train).fillna(0)
pesos_objetivo = pd.cut(freq_objetivo, bins).value_counts(normalize=True)

#ii. Lectores disponibles en test, con su bucket.
disponibles = pd.DataFrame({"id_lector": df_test["id_lector"].drop_duplicates().sort_values()})
disponibles["freq"] = disponibles["id_lector"].map(freq_train).fillna(0)
disponibles["bucket"] = pd.cut(disponibles["freq"], bins)

#iii. Muestreo cada bucket según el peso objetivo (o todo lo que haya, si no alcanza).
n_total = 1000
partes = []
for bucket, peso in pesos_objetivo.items():
    pool = disponibles[disponibles["bucket"] == bucket]
    n_pedido = int(round(peso * n_total))
    partes.append(pool.sample(n=min(n_pedido, len(pool)), random_state=42))

muestra = pd.concat(partes)
lectores_test_muestra = muestra["id_lector"].tolist()

#iv. Comprobación.
print(f"Muestra: {len(lectores_test_muestra)} lectores")
print(pd.DataFrame({
    "objetivo": pesos_objetivo,
    "muestra": muestra["bucket"].value_counts(normalize=True)
}).round(3))
print("\nMediana de frecuencia — muestra:", muestra["freq"].median())
print("Mediana de frecuencia — Kaggle: ", freq_objetivo.median())

Muestra: 1001 lectores
                objetivo  muestra
(0.0, 20.0]        0.281    0.281
(100.0, 250.0]     0.236    0.236
(250.0, inf]       0.173    0.173
(50.0, 100.0]      0.159    0.159
(20.0, 50.0]       0.129    0.129
(-1.0, 0.0]        0.023    0.023

Mediana de frecuencia — muestra: 69.0
Mediana de frecuencia — Kaggle:  72.0


In [ ]:
#c. Calculamos el nDCG@20.
#i. Lista vacía donde iré almacenando los nDCG de cada usuario.
ndcg_lista = []
total_lectores = len(lectores_test_muestra)
print("Comienza la predicción general.")

#ii. Recorro cada id_lector a recomendar.
warnings.filterwarnings("ignore", message=".*eval_at.*")

for i, id_lector in enumerate(lectores_test_muestra, start=1):
    if i % 50 == 0 or i == 1:
        print("Lector {}/{}".format(i, total_lectores))

    #1. Me traigo los libros a recomendarle al id_lector.
    candidatos_brutos = retrieval(id_lector)

    #2. Retrieval en dos etapas: me quedo con los N mejores segun EASE antes de rankear.
    # Ordenar 44.000 items es mucho mas dificil que ordenar 2.000, y EASE ya pone el 59%
    # del GT en su top-2000 (recall medido). Ademas el feature engineering baja de 44.000
    # filas por lector a 2.000, asi que la corrida es ~20 veces mas rapida.
    scores_retrieval = score_ease(
        id_lector, candidatos_brutos, B_ease, libros_idx_ease, leidos_por_lector
    )
    idx_top = np.argsort(-scores_retrieval)[:N_RETRIEVAL]
    libros_candidatos_a_recomendar = [candidatos_brutos[i] for i in idx_top]

    #3. Me traigo el Ground Truth del lector (que por diseño, son 20 en test).
    true_relevance = gt_por_lector[id_lector]

    #4. Realizo el feature engineering sobre todos los libros candidatos a recomendar.
    df_features_candidatos = feature_engineering_test(id_lector, libros_candidatos_a_recomendar)

    #5. Predigo el score de ranking para cada libro candidato del id_lector.
    # OJO: con el ranker esto ya no es un rating predicho (escala 1-10) sino un score de
    # relevancia sin escala interpretable — puede ser negativo. No importa: solo lo uso
    # para ordenar. Pero significa que el "0" del paso 7 ya no es un piso natural.
    predicted_scores_dict = ranking(df_features_candidatos, features, ranker)
   
    #6. Universo en común a evaluar: los libros verdaderos + los que evalué.
    id_libros = list(set(true_relevance.index) | set(predicted_scores_dict.keys()))

    #7. Traigo el rating real de cada libro. Si no es relevante, entonces = 0.
    y_true = np.asarray([[true_relevance.get(id_libro, 0) for id_libro in id_libros]])

    #8. Traigo el score predicho de cada libro. Si no lo evaluó, le pongo el mínimo
    # observado menos 1, para que quede último (con el ranker, 0 podría ser un score alto).
    piso = min(predicted_scores_dict.values()) - 1 if predicted_scores_dict else 0
    y_score = np.asarray([[predicted_scores_dict.get(id_libro, piso) for id_libro in id_libros]])

    #9. Calculo el nDCG@20 para el id_lector.
    ndcg = ndcg_score(y_true, y_score, k=20)

    #10. Lo agrego a la lista.
    ndcg_lista.append(ndcg)

#iii. Imprimo el nDCG promedio de todos los id_lectores.
ndcg_arr = np.array(ndcg_lista)
print("\nEl nDCG@20 promedio es: {:.4f} ± {:.4f} (SE)".format(
    ndcg_arr.mean(), ndcg_arr.std(ddof=1) / np.sqrt(len(ndcg_arr))))
#iv. Exporto los resultados.
np.save("./outputs/ndcg_hibrido.npy", ndcg_arr)

Comienza la predicción general.
Lector 1/1001
Lector 50/1001
Lector 100/1001
Lector 150/1001
Lector 200/1001
Lector 250/1001
Lector 300/1001
Lector 350/1001
Lector 400/1001
Lector 450/1001
Lector 500/1001
Lector 550/1001
Lector 600/1001
Lector 650/1001
Lector 700/1001
Lector 750/1001
Lector 800/1001
Lector 850/1001
Lector 900/1001
Lector 950/1001
Lector 1000/1001

El nDCG@20 promedio es: 0.1223 ± 0.0042 (SE)


#### 10. Análisis.

In [ ]:
#a. Diagnóstico: dónde gana y dónde pierde el modelo.
freq = df_train.groupby("id_lector").size()

res = pd.DataFrame({"id_lector": lectores_test_muestra, "ndcg": ndcg_lista})
res["freq"] = res["id_lector"].map(freq).fillna(0)
res["bucket"] = pd.cut(res["freq"], [-1, 0, 20, 50, 100, 250, np.inf],
                       labels=["cold", "1-20", "21-50", "51-100", "101-250", "250+"])

print(res.groupby("bucket", observed=True)["ndcg"].agg(["mean", "median", "count"]))
print("\nLectores con nDCG = 0:", (res["ndcg"] == 0).mean())

             mean    median  count
bucket                            
cold     0.000000  0.000000     23
1-20     0.154339  0.120317    281
21-50    0.165424  0.129089    129
51-100   0.123320  0.082053    159
101-250  0.102141  0.066138    236
250+     0.080679  0.046788    173

Lectores con nDCG = 0: 0.27672327672327673


In [ ]:
#b. Analizo los del bucket grande y con nDCG = 0 
# (hay dos grupos dentro del grande, los que predicen bien, y los que predicen muy mal. 
# Tengo que investigar si hay patrones).
peores = res[(res["bucket"] == "250+") & (res["ndcg"] == 0)]["id_lector"].head(3)
for lid in peores:
    cands = retrieval(lid)
    fe = feature_engineering_test(lid, cands)
    fe["score"] = ranker.predict(fe[features])
    top = fe.nlargest(20, "score")
    print(f"\n{lid} — leyó {len(leidos_por_lector[lid])} libros")
    print(top[["id_libro", "autor", "prop_lector_autor", "frecuencia_libro"]].head(10))
    print("GT:", list(gt_por_lector[lid].index)[:5])


aladd33 — leyó 500 libros
                                   id_libro              autor  \
44779       fenicias-suplicantes-heraclidas          euripides   
23205                               orestes          euripides   
41286    campos-roturados-tierras-roturadas   sholojov, mijail   
16241                             poemas-14   poe, edgar allan   
32924               cuentos-extraordinarios   poe, edgar allan   
28804               asi-es-si-asi-os-parece  pirandello, luigi   
22449                  el-crucero-del-snark       london, jack   
28913                             andromaca          euripides   
20312                    las-mujeres-de-poe   poe, edgar allan   
35185  andromaca-heracles-loco-las-bacantes          euripides   

       prop_lector_autor  frecuencia_libro  
44779              0.004               0.0  
23205              0.004               0.0  
41286              0.002               1.0  
16241              0.034               1.0  
32924              0.

In [ ]:
# Comentario.
#El modelo encontró que la afinidad por autor predice bien, y su respuesta es: si le gusta Poe, 
# dale todo Poe, incluidas las ediciones que nadie leyó. 
# Pero el ground truth de aladd33 es García Márquez, Cabrera Infante, Philip K. Dick, Flaubert 
# — autores distintos.
#Y explica la bimodalidad del bucket 250+: 
# cuando el autor elegido acierta, el nDCG es alto; cuando no, es exactamente 0. 
# Media 0.045 con mediana 0.000.

In [ ]:
#c. Techo de recall: ¿los 20 libros del GT son siquiera candidatos?
recalls = []
for lid in lectores_test_muestra[:200]:
    cands = set(retrieval(lid))
    gt = set(gt_por_lector[lid].index)
    recalls.append(len(cands & gt) / len(gt))
print("Recall del retrieval:", np.mean(recalls))

Recall del retrieval: 0.9992500000000001


In [ ]:
#d. Comparación pareada contra otra corrida.
# Válido porque el orden de lectores es idéntico entre corridas (muestra estratificada
# con random_state fijo). Detecta diferencias más chicas que comparar promedios sueltos.
from scipy import stats

def comparar(a, b):
    x = np.load(f"./outputs/ndcg_{a}.npy")
    y = np.load(f"./outputs/ndcg_{b}.npy")
    print(f"{a}: {x.mean():.4f}  ->  {b}: {y.mean():.4f}   (dif {y.mean()-x.mean():+.4f})")
    print("  ", stats.wilcoxon(x, y))

# comparar("rf_base", "lgbmranker_base")

In [ ]:
#e. Comentario.
#Para un lector fijo, el score de un candidato lo deciden rating_prom_id_lector_autor, 
# n_interacciones_lector_autor y las dos de género. 
# Un lector con 15 libros leyó quizá 12 autores: el modelo tiene que elegir entre "autor que 
# ya leyó" (12 candidatos, señal fuerte) y "autor desconocido" 
# (todo el resto, n=0 y rating=media_global). 
# Es una decisión casi binaria y acierta seguido, porque la gente vuelve a sus autores.

#Un lector con 200 libros leyó 150 autores. Ahora hay 150 candidatos con señal positiva y el modelo
#  tiene que ordenarlos entre sí. 
# Eso ya no lo resuelve n_interacciones_lector_autor, que satura. 
# Y las features que podrían desempatar —calidad del libro, popularidad, novedad— son las tres que
#  tenés comentadas.

#O sea: tus features distinguen bien "conocido vs desconocido" y no distinguen nada dentro de 
# "conocido". El lector pesado vive enteramente dentro de esa segunda categoría.

# Y justamente en el dataset a predecir, la mayoría son personas con muchos libros leídos.
# Osea, los estoy subestimando.

In [ ]:
############## VERSIÓN BASE:
# NO SE TRABABA LOS CASOS DONDE HABIA LIBROS Y LECTORES CON INTERACCIONES QUE NO ESTABAN EN
# DF_LECTORES NI DF_LIBROS.... A REVISAR.

# NO ESTAN CALCULADAS LAS COLUMNAS DE PROPORCIONES NI POPULARIDAD.
# A TENER EN CUENTA.